In [ ]:
""" !pip install -r requirements.txt """

# Import Libraries

In [ ]:
import os
import time
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    PredefinedSplit,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OneHotEncoder
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import plot_tree

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    roc_curve,
    PrecisionRecallDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from plot import *
from utils import *


---
# Support functions

# Global Variables

In [ ]:
seed = 42

FILENAME = "../../data/train.csv"

# features = []

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df2 = df.T.drop_duplicates().T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


## 2. Label extraction

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
print(X['hardship_type_label'].unique())
print(X['hardship_loan_status_label'].unique())
print(X['hardship_status_label'].unique())

## 3. Data visualization

In [ ]:
plot_feature_distribution(y, "grade")
plot_nan(X)

## 4. Features manipulation

In [ ]:
for i in X.dtypes.unique():
    print("Type", i)

In [ ]:
# Teniamo una flag per indicare se un hardship loan e' stato concesso o meno
# X['has_hardship_plan'] = X['hardship_start_date'].notna().astype(float)

In [ ]:
X = drop_high_nan_columns(X)

### Categorical features

In [ ]:
##### FEATURES EXTRACTION ######
# Trasforma "36 months" e "60 months" in float type
print("\n", X['loan_contract_term_months'].unique())
X['loan_contract_term_months'] = X['loan_contract_term_months'].str.extract(r'(\d+)').astype(float)

# Strip della stringa "years"
# Trasforma anni in float: < 1 diventa 0, 10+ diventa 10
print("\n", X['loan_contract_term_months'].unique())
X['borrower_profile_employment_length'] = X['borrower_profile_employment_length'].str.replace(r'\+? years?', '', regex=True)
X['borrower_profile_employment_length'] = X['borrower_profile_employment_length'].replace({ '< 1': 0}).astype(float)


##### DROP FEATURES #####
categorical_to_drop = [
  'loan_title',                         # non significant column, grande sparsita' di dati. Sufficiente loan_purpose_category come aggregazione di scopo del prestito
  'borrower_address_zip',               # non significant column, esiste una colonna per identificazione stati
]

X = X.drop(columns=categorical_to_drop)

### Numerical features

In [ ]:
# 1. DROP: Data Leakage (Future Information)
numerical_to_drop = [
    # Future income for loaner
    'total_payment_received', 'total_received_principal', 'recoveries_cash',
    'collection_recovery_fee', 'last_payment', 'outstanding_principal_balance',
    'total_received_interest', 'total_received_late_fees',
]

X = X.drop(columns=numerical_to_drop)


print("Nuovo # Colonne: " +  str(X.shape[1]) + "\n")

In [ ]:
# 2. DROP: Secondary/Joint Applicant info
# application_type_label ci informa gia' se il tipo di prestito e' individual o

joint_and_secondary_cols = [col for col in X.columns if col.startswith('joint_') or col.startswith('secondary_')]

# DROP: tutti i campi relativi a settlement sono data leakage
settlement_cols = [col for col in X.columns if 'settlement' in col]

to_drop = joint_and_secondary_cols + settlement_cols

X = X.drop(columns=to_drop)

print("Nuovo # Colonne: " +  str(X.shape[1]) + "\n")

In [ ]:
# Feature categoriche in cui i valori NaN sono riempiti con una nuova label Unknown
categorical_to_unknown = [
  'borrower_address_state',
  'loan_purpose_category',
  'loan_status_current_code',
  'borrower_income_verification_status',
  'borrower_housing_ownership_status'
]

# 2. Fill with 'unknown'
for col in categorical_to_unknown:
    X[col] = X[col].fillna('unknown')


# 3. KEEP & FILL: "Months Since" columns (NaN = Never happened)
# We fill with a large number (e.g., 100 months) to signify "very long time ago / never"
structural_cols = [
    'months_since_last_public_record',
    'months_since_recent_bankcard_delinquency',
    'months_since_last_major_derog',
    'months_since_recent_revolving_delinquency',
    'months_since_last_delinquency'
]

for col in structural_cols:
    X[col] = X[col].fillna(100)


In [ ]:
##### DATES #####
X['loan_issue_date'] = pd.to_datetime(X['loan_issue_date'], format='%b-%Y')

X['issue_month'] = X['loan_issue_date'].dt.month
X['issue_year'] = X['loan_issue_date'].dt.year

# cyclical Encoding
X['issue_month_sin'] = np.sin(2 * np.pi * X['issue_month'] / 12)
X['issue_month_cos'] = np.cos(2 * np.pi * X['issue_month'] / 12)

# calculate
X['credit_history_earliest_line'] = pd.to_datetime(X['credit_history_earliest_line'], format='%b-%Y')

# Numero mesi passati tra prima richiesta credito e loan date
X['months_since_earliest_cr_line'] = (
    (X['loan_issue_date'].dt.year - X['credit_history_earliest_line'].dt.year) * 12 +
    (X['loan_issue_date'].dt.month - X['credit_history_earliest_line'].dt.month)
)


# Tutte le features rimanenti con date non sono rilevanti (leakage)
date_cols = [col for col in X.columns if 'date' in col]
to_drop = [
    'credit_history_earliest_line',     # used for feature extraction
    'issue_month',                      # used for feature extraction: sin/cos encoding
    ] + date_cols


X = X.drop(columns=to_drop)
print(f"Columns remaining: {X.shape[1]}")

## NaN management

In [ ]:
print_nan(X)

In [ ]:
# Split train test: 0.25
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)

In [ ]:
# Feature categoriche in cui i valori NaN possono essere riempiti con la moda:
# valori con 2 etichette, in cui la mancanza di un dato viene trattato come "no" o come occorrenza piu' frequente
fill_to_mode =[
  # Categorical
  'disbursement_method_type',
  'application_type_label',
  'listing_initial_status',
  'loan_payment_plan_flag',
  'hardship_flag_indicator',      # n/y
  'loan_payment_plan_flag',       # n/y

  # Numerical
  'loan_contract_term_months'  # 36/60 mesi
]

for i in fill_to_mode:
    print(X[i].unique())


# 1. Setup the imputer with 'most_frequent' (Mode)
mode_imputer = SimpleImputer(strategy='most_frequent')

# 2. Fit on TRAIN only (Learn the most common categories)
# It's important to select only these columns to avoid errors with other types
mode_imputer.fit(X_train[fill_to_mode])

# 3. Transform BOTH (Fill NaNs)
# We assign the result back to the specific columns to keep the DataFrame structure
X_train[fill_to_mode] = mode_imputer.transform(X_train[fill_to_mode])
X_val[fill_to_mode] = mode_imputer.transform(X_val[fill_to_mode])

In [ ]:

""" numerical_to_mean = [
    'borrower_dti_ratio',
    'borrower_income_annual',
    'revolving_utilization',
    'total_revolving_high_credit_limit',
] """

numerical_to_median = X_train.select_dtypes(include=['float', 'int']).columns

# Setup the imputer
# Fill with median values to not influence skewed data
imputer = SimpleImputer(strategy='median')

# Fit on TRAIN only (Learn the medians)
imputer.fit(X_train[numerical_to_median])

# Transform BOTH (Fill NaNs)
X_train[numerical_to_median] = imputer.transform(X_train[numerical_to_median])
X_val[numerical_to_median] = imputer.transform(X_val[numerical_to_median])

In [ ]:
print(X_train['hardship_flag_indicator'].unique())

In [ ]:
""" print_nan(X_train)
plot_nan(X_train)

print_nan(X_val)
plot_nan(X_val) """

In [ ]:
""" num_cols = X_train.select_dtypes(include=['float64', 'int64'])
analyze_feature_distributions(num_cols, save_plots=True, output_folder="grafici_explorativi") """

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

def get_distribution_type(data, threshold_skew=1.0, threshold_peaks=0.05):
    """
    Funzione helper per identificare il tipo di distribuzione di una singola Series.
    """
    # Rimuoviamo NaN per l'analisi
    clean_data = data.dropna()
    if len(clean_data) < 50: return 'Other'
    
    # 1. Calcolo Skewness
    skew_val = clean_data.skew()
    
    # 2. Test Normalità (D'Agostino's K-squared test)
    try:
        k2, p_norm = stats.normaltest(clean_data)
        # Se p > 0.01 e skewness è bassa, consideriamo "Normale"
        is_normal = (p_norm > 0.01) and (abs(skew_val) < 0.5)
    except:
        is_normal = False

    if is_normal:
        return 'Normal'

    # 3. Controllo Skewness (Priorità alta)
    if abs(skew_val) > threshold_skew:
        return 'Skewed'

    # 4. Controllo Multimodalità (Picchi)
    counts, bin_edges = np.histogram(clean_data, bins='auto', density=True)
    peaks, _ = find_peaks(counts, prominence=np.max(counts) * threshold_peaks)
    
    if len(peaks) > 1:
        return 'Multimodal'
        
    return 'Other' # Simmetrica ma non normale, o altro

def auto_transform_features(df):
    """
    Analizza ogni colonna numerica e applica la trasformazione:
    - Skewed -> Log Transformation
    - Normal -> Z-Score Standardization
    - Multimodal -> Binning (5 Quantiles)
    """
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    
    # Report per tenere traccia delle modifiche
    report_list = []
    
    for col in num_cols:
        # Identifica la distribuzione
        dist_type = get_distribution_type(df_clean[col])
        action = "Nessuna azione"
        
        # --- LOGICA DI TRASFORMAZIONE ---
        
        if dist_type == 'Skewed':
            # LOG TRANSFORMATION
            # Gestione valori negativi/zero: trasliamo se necessario
            min_val = df_clean[col].min()
            if min_val <= 0:
                offset = abs(min_val) + 1
                df_clean[col] = np.log(df_clean[col] + offset)
                action = f"Log Transform (Shifted +{offset})"
            else:
                df_clean[col] = np.log(df_clean[col])
                action = "Log Transform"
                
        elif dist_type == 'Normal':
            # Z-SCORE STANDARDIZATION
            # Reshape necessario per StandardScaler (n_samples, 1)
            values = df_clean[col].values.reshape(-1, 1)
            df_clean[col] = scaler.fit_transform(values)
            action = "Z-Score Standardization"
            
        elif dist_type == 'Multimodal':
            # BINNING
            # Usiamo qcut per dividere in 5 quantili (0,1,2,3,4)
            # duplicates='drop' gestisce casi in cui molti valori sono identici
            try:
                df_clean[col] = pd.qcut(df_clean[col], q=5, labels=False, duplicates='drop')
                action = "Binning (5 Quantiles)"
            except Exception as e:
                action = f"Binning Fallito: {e}"
        
        # Salviamo il report
        report_list.append({
            'Feature': col,
            'Tipo Rilevato': dist_type,
            'Trasformazione': action
        })
    
    report_df = pd.DataFrame(report_list)
    return df_clean, report_df



In [ ]:
""" num_cols = X_train.select_dtypes(include=['float64', 'int64'])
df_transformed, report = auto_transform_features(num_cols)

print("Riepilogo Trasformazioni:")
print(report.head(20)) """

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import find_peaks

def identify_distributions(df, threshold_skew=1.0, threshold_peaks_prominence=0.05):
    """
    Identifica se le colonne numeriche sono Skewed, Multimodali, Uniformi o Normali.
    Restituisce un DataFrame con i risultati dell'analisi.
    """
    results = []
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in num_cols:
        data = df[col].dropna()
        if len(data) < 50: continue # Salta colonne con troppi pochi dati
            
        # 1. Check Skewness (Asimmetria)
        skew_val = data.skew()
        is_skewed = abs(skew_val) > threshold_skew
        
        # 2. Check Normality (Test di D'Agostino's K-squared)
        try:
            k2, p_norm = stats.normaltest(data)
            # p_value > 0.01 e skewness bassa indicano normalità
            is_normal = (p_norm > 0.01) and (abs(skew_val) < 0.5)
        except:
            is_normal = False
            
        # 3. Check Multimodality (Picchi nell'istogramma)
        # Calcola la densità approssimata tramite istogramma
        counts, bin_edges = np.histogram(data, bins='auto', density=True)
        # Trova picchi che siano rilevanti (almeno il 5% della densità massima)
        peaks, _ = find_peaks(counts, prominence=np.max(counts) * threshold_peaks_prominence)
        num_peaks = len(peaks)
        is_multimodal = num_peaks > 1
        
        # 4. Check Uniformity (Varianza delle frequenze)
        # Se la deviazione standard delle frequenze nei bin è molto bassa, è uniforme
        counts_uni, _ = np.histogram(data, bins=20) 
        cv = np.std(counts_uni) / np.mean(counts_uni)
        is_uniform = cv < 0.2 # Soglia euristica: deviazione < 20% della media

        # Logica di Classificazione (Priorità)
        classification = "Unknown"
        if is_uniform:
            classification = "Uniform"
        elif is_multimodal and not is_skewed: 
             # Spesso le code lunghe creano falsi picchi, quindi diamo priorità allo skew
             classification = "Multimodal"
        elif is_skewed:
            classification = "Positively Skewed" if skew_val > 0 else "Negatively Skewed"
        elif is_normal:
            classification = "Normal"
        else:
             classification = "Symmetric (Non-Normal)"

        results.append({
            'Feature': col,
            'Skewness': round(skew_val, 2),
            'Num_Peaks': num_peaks,
            'Classification': classification
        })
        
    return pd.DataFrame(results)

In [ ]:
""" aaa = identify_distributions(X_train)
print(aaa.to_string()) """

In [ ]:
def apply_capping(df, lower_quantile=0.01, upper_quantile=0.99):
    """
    Applica la tecnica del Capping (Winsorization) alle colonne numeriche di un DataFrame.
    I valori sotto il lower_quantile vengono sostituiti con il valore del lower_quantile.
    I valori sopra l'upper_quantile vengono sostituiti con il valore dell'upper_quantile.
    
    Parametri:
    - df: DataFrame in input
    - lower_quantile: soglia inferiore (default 0.01 -> 1%)
    - upper_quantile: soglia superiore (default 0.99 -> 99%)
    
    Return:
    - DataFrame con i valori cappati.
    """
    # Creiamo una copia per non modificare l'originale in-place
    df_capped = df.copy()
        
    # Selezioniamo solo le colonne numeriche
    numerical_cols = df_capped.select_dtypes(include=['float', 'int']).columns
    
    for col in numerical_cols:
        # Calcolo dei limiti
        lower_limit = df_capped[col].quantile(lower_quantile)
        upper_limit = df_capped[col].quantile(upper_quantile)
        
        # Applicazione del capping (clipping)
        # I valori < lower_limit diventano lower_limit
        # I valori > upper_limit diventano upper_limit
        df_capped[col] = df_capped[col].clip(lower=lower_limit, upper=upper_limit)
        
        # Opzionale: Stampa per vedere l'effetto (puoi commentarlo)
        print(f"Colonna '{col}': cappata tra {lower_limit:.2f} e {upper_limit:.2f}")
        
    return df_capped


X_train = apply_capping(X_train)

In [ ]:
num_cols = X_train.select_dtypes(include=['float64', 'int64'])
analyze_feature_distributions(num_cols, save_plots=False, output_folder="grafici_explorativi")

## Features encoding

In [ ]:

categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Create encoder with handle_unknown to deal with unseen categories
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data only
ohe.fit(X_train[categorical_features])

# Transform both sets
X_train_encoded = ohe.transform(X_train[categorical_features])
X_val_encoded = ohe.transform(X_val[categorical_features])

# Get feature names for the encoded columns
encoded_feature_names = ohe.get_feature_names_out(categorical_features)

# Create DataFrames with the encoded features
X_train_ohe = pd.DataFrame(X_train_encoded,
                           columns=encoded_feature_names,
                           index=X_train.index)
X_val_ohe = pd.DataFrame(X_val_encoded,
                         columns=encoded_feature_names,
                         index=X_val.index)

# Drop original categorical columns and concatenate encoded ones
X_train = X_train.drop(columns=categorical_features).join(X_train_ohe)
X_val = X_val.drop(columns=categorical_features).join(X_val_ohe)


le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)


print(X_train.shape[1])
print(X_train.head())

In [ ]:
plot_feature_distribution(y_train, "EEEEEEEEEEEEEEEEEEEE")

# Training

## Define the scalers to be used

In [ ]:
# Scalers to test
scalers = {
    "MinMaxScaler": MinMaxScaler(),
    "StandardScaler": StandardScaler()
}

## Apply Random Forests with hyperparameters tuning

In [ ]:
param_grid_rf = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_features': ['sqrt', 'log2'],
    'clf__criterion': ['gini', 'entropy', 'log_loss'],
    'clf__max_depth': [None, 10, 20]
}

pipeline = Pipeline([
    ("clf", RandomForestClassifier(random_state=seed, n_jobs=-1, class_weight='balanced'))        #No oversampling needed in random forest (?)
])

grid = GridSearchCV(pipeline, param_grid_rf, cv=5, scoring='balanced_accuracy', n_jobs=-1, verbose=3)
grid.fit(X_train, y_train)

print("Performance:", grid.best_estimator_)
print("Performance:", grid.best_params_)

with open("rf.save","wb") as file:
    pickle.dump(grid.best_estimator_['clf'], file)

In [ ]:
evaluate_model("rf.save", X_val, y_val, "Random Forest (Best)")

In [ ]:
visualize_rf_tree("rf.save", X_train, y_train, max_depth=5)

In [ ]:
plot_top_correlations_split(X_train, y_train)

## Apply Support Vector Classifier with hyperparameters tuning

In [ ]:
# SVC param grid
# L' uso di LinearSVC su classic SVC (inserire differenza) e' necessario causa n_sample esteso (si consiglia se > 10k-ish)
param_grid_svc = {
    'clf__C': [20],      #[0.1, 1, 10, 100, 1000],
    'clf__kernel': ["rbf"]     #, "poly"
    }

""" param_grid_lsvc = {
    'clf__C': [30, 40, 50, 100],
    'clf__max_iter': [250, 500, 750]
} """

# Grid Search with Pipeline
best_score = 0
for scaler_name, scaler in scalers.items():

        pipeline = Pipeline([
            # 1. Scale first (LinearSVC needs it)
            ('scaler', scaler),

            # 2. SMOTE inside the fold (Prevents leakage!)
            #('smote', SMOTE(random_state=seed)),

            ("clf", SVC(class_weight="balanced", random_state=seed))

            # 3. Use Fast LinearSVC
            #('clf', LinearSVC(dual=False, loss='squared_hinge', class_weight='balanced', random_state=seed))             #Prefer dual=False when n_samples > n_features
        ])

        # Perform Grid Search
        grid =  GridSearchCV(pipeline, param_grid_svc, cv=5, scoring='balanced_accuracy', n_jobs=-1, verbose=3)
        grid.fit(X_train, y_train)


        print("Performance:", grid.best_estimator_)
        print("Performance:", grid.best_params_)


        if grid.best_score_ > best_score:
            best_score = grid.best_score_
            with open("svc_scaler.save","wb") as file:
                pickle.dump(grid.best_estimator_['scaler'], file)
            with open("svc.save","wb") as file:
                pickle.dump(grid.best_estimator_['clf'], file)

In [ ]:
evaluate_model("svc.save", X_val, y_val, "SVC")

## Apply K Neighbors Classifier with hyperparameters tuning

In [ ]:
# KNN param grid
param_grid_knn = {
    'clf__n_neighbors': [5, 15, 30],
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan']
}
# Grid Search with Pipeline
best_score = 0
for scaler_name, scaler in scalers.items():

        pipeline = ImbPipeline([
            ('scaler', scaler),
            ('smote', SMOTE(random_state=seed)),
            ('clf', KNeighborsClassifier(n_jobs=-1))
        ])

        # Perform Grid Search
        grid =  GridSearchCV(pipeline, param_grid_knn, cv=5, scoring='balanced_accuracy', n_jobs=-1, verbose=3)
        grid.fit(X_train, y_train)


        print("Performance:", grid.best_estimator_)
        print("Performance:", grid.best_params_)


        if grid.best_score_ > best_score:
            best_score = grid.best_score_
            with open("knn_scaler.save","wb") as file:
                pickle.dump(grid.best_estimator_['scaler'], file)
            with open("knn.save","wb") as file:
                pickle.dump(grid.best_estimator_['clf'], file)

In [ ]:
evaluate_model("knn.save", X_val, y_val, "KNN")

# NN Torch

In [ ]:
# look for GPU
# controllare device usato per il parallelismo

if torch.backends.mps.is_available():
    print("MPS device is available.")       # Apple silicon
    device = torch.device("mps")
elif torch.cuda.is_available():             # Nvidia
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.") # Se nessuno disponibile, usa CPU
    device = torch.device("cpu")

In [ ]:
# Define the Data Layer

class MyDataset(Dataset):
    def __init__(self, X, y):

        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

        self.num_samples = X.shape[0]         # Definisco numero di samples
        self.num_features = X.shape[1]        # Definisco numero di feature
        self.num_classes = len(np.unique(y))  # Definisco numero di classi


    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.X[idx, :], self.y[idx]

In [ ]:
# For reproducibility

def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True  # slower, versione esatta completa di ogni algoritmo, senza approssimazioni

In [ ]:
def test_model(model, data_loader, device):
    model.eval()
    y_pred = []
    y_test = []

    for data, targets in data_loader:
        data, targets = data.to(device), targets.to(device)
        y_pred += model(data)
        y_test += targets

    y_test = torch.stack(y_test).squeeze()
    y_pred = torch.stack(y_pred).squeeze()
    y_pred_c = y_pred.argmax(dim=1, keepdim=True).squeeze()

    return y_test, y_pred_c, y_pred

In [ ]:
# Define a function for the training process
# Parametri:
## modello,
## loss (cross entropy)
## ottimizzatore (sgd, adam...)
## numero di epoche
## learning scheduler
## train, validation e test loader: prendono singoli dataset e producono batch
## device su cui lavoriamo
## writer di log
## log

def train_model(model, criterion, optimizer, n_epochs, scheduler, train_loader, val_loader, device, writer, log_name="best_model"):
    n_iter = 0
    best_valid_loss = float('inf')
    for ep in range(n_epochs):
        model.train()

        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)

            optimizer.zero_grad()

            # Forward pass
            y_pred = model(data)

            # Compute Loss
            loss = criterion(y_pred, targets)
            writer.add_scalar("Loss/train", loss, n_iter)

            # Backward pass
            loss.backward()
            optimizer.step()

            n_iter += 1

        # --- VALIDATION PHASE ---
        # test_model restituisce (labels_reali, predizioni_classi, probabilità_output)
        labels_val, preds_val, y_output_val = test_model(model, val_loader, device)

        # 1. Calcolo Loss di Validation
        loss_val = criterion(y_output_val, labels_val)
        writer.add_scalar("Loss/val", loss_val, ep)

        # 2. CALCOLO ACCURACY DI VALIDATION
        # Calcoliamo quante predizioni sono uguali alle label reali
        correct = (preds_val == labels_val).sum().item()
        total = labels_val.size(0)
        acc_val = 100 * correct / total

        # Log dell'accuracy su TensorBoard
        writer.add_scalar("Accuracy/val", acc_val, ep)

        print(f"Epoch {ep+1}/{n_epochs} - Loss Val: {loss_val:.4f} - Acc Val: {acc_val:.2f}%")

        # Save best model basandosi sulla loss (o potresti usare acc_val)
        if loss_val.item() < best_valid_loss:
            best_valid_loss = loss_val.item()
            if not os.path.exists('models'):
                os.makedirs('models')
            torch.save(model.state_dict(), 'models/'+log_name)

        writer.add_scalar("Learning Rate", scheduler.get_last_lr()[0], ep)

        scheduler.step()

    return model

In [ ]:
print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

In [ ]:
# Scale data
# Preprocessing: Scale data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Create the torch dataset
train_subset = MyDataset(X_train,y_train)
val_subset = MyDataset(X_val,y_val)

In [ ]:
# Let's define a new architecture inlcuding also Dropout
class FeedForward_OurNN(nn.Module):
    def __init__(self, input_size, num_classes, hidden_size, dropout_rate, depth=1):
        super(FeedForward_OurNN, self).__init__()

        model = [
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            #nn.Dropout(p=dropout_rate)
        ]

        block = [
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            #nn.Dropout(p=dropout_rate)
        ]

        for i in range(depth):
            model += block

        self.model = nn.Sequential(*model)

        self.output = nn.Linear(hidden_size, num_classes)


    def forward(self, x):
        h = self.model(x)
        out = self.output(h)
        return out

In [ ]:
# fixed hyperparameters
num_epochs = 50
learning_rate = 0.01
gamma = 0.5
step_size = 10

# hyperparameter to validate
batch_size = 1024
hidden_size = 8
depth = 1
dropout_rate = 0.5 # new hyperparameter

# Cycle over hyperparameters training a new model every time, saving the best model and validating it

In [ ]:
# fix the seed for reproducibility

 ## mettere il seguente in un ciclo da iterare su
 # itertools per runnare sui vari parametri

fix_random(seed)


# Start tensorboard
exp_name = "runs/FeedForward_OurNN_depth" + str(depth) + "_hidden" + str(hidden_size) + "_dropout" + str(dropout_rate)
writer = SummaryWriter(log_dir=exp_name)


# Create relative dataloaders
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size)


# Define the architecture, loss and optimizer
model = FeedForward_OurNN(train_subset.num_features, train_subset.num_classes, hidden_size, dropout_rate, depth)
print(model)
model.to(device)

# Add model graph to TensorBoard
data_sample, _ = next(iter(train_loader))
writer.add_graph(model, data_sample.to(device))

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
#optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)


# Test before the training
#y_test, y_pred_c, _ = test_model(model, test_loader, device)
#acc = (y_test == y_pred_c).float().sum() / y_test.shape[0]
#print("Accuracy before training:", acc.cpu().numpy())


# Train the model
model = train_model(model, criterion, optimizer, num_epochs, scheduler, train_loader, val_loader, device, writer)


# Load best model
model.load_state_dict(torch.load("models/best_model", weights_only=True))
model.to(device)


# Test after the training
#y_test, y_pred_c, _ = test_model(model, test_loader, device)
#acc = (y_test == y_pred_c).float().sum() / y_test.shape[0]
#print("Accuracy after training:", acc.cpu().numpy())


# Close tensorboard writer after a training
writer.flush()
writer.close()

In [ ]:
# --- 1. Definizione della Griglia degli Iperparametri ---
param_grid = {
    'batch_size': [512],
    'hidden_size': [128],
    'depth': [1, 2],
    'dropout_rate': [0.2]
}

# Parametri fissi
num_epochs = 50
learning_rate = 0.01
gamma = 0.5
step_size = 10
seed = 42

# Genera tutte le combinazioni possibili
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_val_acc_overall = 0.0
best_config = None

# --- 2. Ciclo di Iterazione ---
for i, config in enumerate(combinations):
    # Estrazione parametri correnti
    bs = config['batch_size']
    hs = config['hidden_size']
    d = config['depth']
    dr = config['dropout_rate']

    print(f"\n[Run {i+1}/{len(combinations)}] Testing: {config}")

    # Fix the seed for reproducibility ad ogni inizio ciclo
    fix_random(seed)

    # Start tensorboard con nome esperimento dinamico
    exp_name = f"runs/FFNN_bs{bs}_d{d}_h{hs}_dr{dr}"
    writer = SummaryWriter(log_dir=exp_name)

    # Create relative dataloaders
    train_loader = DataLoader(train_subset, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=bs)

    # Define the architecture, loss and optimizer
    # Assicurati che l'ordine degli argomenti in FeedForward_OurNN sia corretto
    model = FeedForward_OurNN(train_subset.num_features, train_subset.num_classes, hs, dr, d)
    model.to(device)

    # Add model graph to TensorBoard (solo alla prima iterazione per non appesantire)
    if i == 0:
        data_sample, _ = next(iter(train_loader))
        writer.add_graph(model, data_sample.to(device))

    criterion = torch.nn.CrossEntropyLoss()
    #optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    # Nome univoco per il miglior modello di QUESTA specifica run
    current_run_model_name = f"best_model_run_{i}"

    # Train the model
    # Nota: la tua funzione train_model salva già internamente il miglior modello della run in 'models/'
    model = train_model(model, criterion, optimizer, num_epochs, scheduler,
                        train_loader, val_loader, device, writer, log_name=current_run_model_name)

    # --- 3. Validazione Finale e Confronto ---
    # Carichiamo il miglior stato salvato durante QUESTO specifico addestramento
    model.load_state_dict(torch.load(f"models/{current_run_model_name}", weights_only=True))
    model.to(device)

    # Test sul validation set per l'accuratezza finale
    labels_val, preds_val, _ = test_model(model, val_loader, device)
    acc = (preds_val == labels_val).float().mean().item()

    print(f"Accuracy per questa config: {acc:.4f}")

    # Log dell'accuratezza finale su TensorBoard (sezione Hparams)
    writer.add_hparams(config, {"hparam/accuracy": acc})

    # Salvataggio del modello migliore in assoluto di tutta la Grid Search
    if acc > best_val_acc_overall:
        best_val_acc_overall = acc
        best_config = config
        torch.save(model.state_dict(), "models/BEST_OVERALL_MODEL.pth")
        print("!!! Nuovo miglior modello assoluto trovato !!!")

    # Close tensorboard writer
    writer.flush()
    writer.close()

print("\n" + "="*40)
print(f"GRID SEARCH TERMINATA")
print(f"Miglior accuratezza: {best_val_acc_overall:.4f}")
print(f"Miglior configurazione: {best_config}")